# VirtualCraft GUI Texture Playground

Interactive pixel-debugging companion to `gui_texture_generator.py`.

**Workflow:**
1. Edit a color constant in `Palette` or a coordinate in `Machines.xxx` (in `gui_texture_generator.py`).
2. Re-run the cell below to see the result live (auto-reload picks up the change).
3. When happy, call `save_png(...)` or run `python gui_texture_generator.py` from a terminal to write to disk.

**Tips:**
- `show(arr, scale=4)` zooms with nearest-neighbour so individual pixels are crisp.
- `diff_overlay(a, b)` paints differing pixels in magenta — great for verifying tweaks.
- All `Machines.xxx()` return a `numpy.ndarray` of shape `(H, W, 4)` (RGBA, uint8).

In [ ]:
# Auto-reload the generator module on every edit (no kernel restart needed)
%load_ext autoreload
%autoreload 2

import importlib
import sys
from pathlib import Path

# The notebook lives at tools/python/notebook_playground.ipynb. Add that dir
# to sys.path regardless of where Jupyter was launched from (project root, tools/, or here).
PYTHON_DIR = None
for candidate in [
    Path('.').resolve(),
    Path('.').resolve() / 'tools' / 'python',
    Path('tools') / 'python',
    Path('..'),
]:
    candidate = candidate.resolve()
    if (candidate / 'gui_texture_generator.py').exists():
        PYTHON_DIR = candidate
        break
if PYTHON_DIR is None:
    raise RuntimeError(
        'Could not locate gui_texture_generator.py — '
        'open Jupyter from the project root, tools/, or tools/python/.'
    )
if str(PYTHON_DIR) not in sys.path:
    sys.path.insert(0, str(PYTHON_DIR))

import gui_texture_generator as G
importlib.reload(G)  # safety net if autoreload missed a change

import numpy as np
from PIL import Image
from IPython.display import display

print(f'Loaded {G.__file__}')
print(f'OUT_DIR  = {G.OUT_DIR}')
print(f'ICONS_PATH = {G.ICONS_PATH}')
print(f'Generators: {list(G.GENERATORS.keys())}')

## Display helpers

In [ ]:
def show(arr, scale: int = 1, crop=None):
    """Display a numpy RGBA array.

    Parameters
    ----------
    arr : np.ndarray (H, W, 4)
    scale : int
        Nearest-neighbour zoom factor. Useful for pixel art (scale=4..8).
    crop : tuple (x, y, w, h) or None
        Optional crop before display.
    """
    if crop is not None:
        x, y, w, h = crop
        arr = arr[y:y + h, x:x + w]
    img = Image.fromarray(arr, 'RGBA')
    if scale != 1:
        img = img.resize((img.width * scale, img.height * scale), Image.NEAREST)
    display(img)


def diff_overlay(a, b, scale: int = 4):
    """Display `a` with differing pixels (vs `b`) highlighted magenta.

    Returns the diff count so you can assert in code.
    """
    assert a.shape == b.shape, f'shape mismatch: {a.shape} vs {b.shape}'
    diff_mask = np.any(a != b, axis=-1)
    n = int(np.sum(diff_mask))
    overlay = a.copy()
    overlay[diff_mask] = (255, 0, 255, 255)  # magenta
    print(f'{n} pixel(s) differ')
    show(overlay, scale=scale)
    return n

## 1. PowMiner 176×178

- ENERGY (8, 22) 16×52 vertical bar
- OUTPUT (79, 32) 18×18 single slot
- ARROW (62, 56) 52×8 purple horizontal bar

In [ ]:
arr = G.Machines.pow_miner()
show(arr, scale=2, crop=(0, 0, 180, 180))

## 2. ElectricBallGenerator 176×178

- FUEL (47, 39) 18×18 left input
- ARROW (77, 40) 22×16 flame arrow (empty -> orange when burning)
- ENERGY (112, 22) 16×52 right energy buffer (v20.15 — vertical bar, NOT slot)

In [ ]:
arr = G.Machines.electric_ball_generator()
show(arr, scale=2, crop=(0, 0, 180, 180))

## 3. ElectricCondenser 176×178

All widgets centered on y=49. UPGRADE column at right.

In [ ]:
arr = G.Machines.electric_condenser()
show(arr, scale=2, crop=(0, 0, 180, 180))

## 4. SideConfigMenu 120×107 (blank backdrop)

In [ ]:
arr = G.Machines.side_config_menu()
show(arr, scale=3, crop=(0, 0, 124, 111))

## 5. icons.png 256×256

Full sheet with palette + button atlas + vertical FE fill + hand-drawn icons.

In [ ]:
arr = G.Machines.icons_sheet()
show(arr, scale=2)

### Zoom on individual icon cells (debug your hand-drawn pixel art)

In [ ]:
# 16×16 icons row @ y=40..55 — show each cell at 12x zoom for pixel-by-pixel inspection
for label, x in [('gear', 0), ('back', 16), ('input', 48), ('output', 64), ('disable', 80)]:
    print(f'--- {label} @ ({x}, 40) ---')
    show(arr, scale=12, crop=(x, 40, 16, 16))

## Regression check: do all 5 textures match what's currently on disk?

In [ ]:
for name, fn in G.GENERATORS.items():
    G.diff_png(fn(), f'{name}.png')

## Save (writes to `src/main/resources/.../gui/`)

Uncomment one of the lines below — or run `python gui_texture_generator.py` from a terminal for the same effect.

In [ ]:
# G.save_png(G.Machines.electric_ball_generator(), 'electric_ball_generator.png')
# G.save_png(G.Machines.icons_sheet(), 'icons.png')
# for name, fn in G.GENERATORS.items():
#     G.save_png(fn(), f'{name}.png')

## Live palette editing example

Demonstrates the fast feedback loop without touching the source file. The change is **local to this notebook** — it does NOT modify `gui_texture_generator.py`.

In [ ]:
# Snapshot the original color so we can restore it
original_purple = G.Palette.ACCENT_PURPLE

# Try a brighter neon-cyan progress arrow instead of the canonical purple
G.Palette.ACCENT_PURPLE = G._c(0x00E5FF)
G.Palette.ICON_PURPLE_LIGHT = G._c(0x80F8FF)
G.Palette.ACCENT_VIOLET = G._c(0x0099B8)

show(G.Machines.electric_condenser(), scale=3, crop=(178, 0, 24, 18))  # zoom on the arrow sprite

# Restore — don't pollute subsequent cells
G.Palette.ACCENT_PURPLE = original_purple
G.Palette.ICON_PURPLE_LIGHT = G._c(0xA79CFF)
G.Palette.ACCENT_VIOLET = G._c(0x6054A6)